In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
df = pd.read_csv('/content/drive/MyDrive/youtube_shorts.csv')

In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [29]:
print("First 5 rows:")
display(df.head())

print("\nInfo:")
print(df.info())


First 5 rows:


,video_id,title,duration_sec,hashtags_count,views,likes,comments,shares,upload_hour,category
0,vid_1000,Short Video #0,43,9,198775,21933,3228,400,8,Tech
1,vid_1001,Short Video #1,56,2,290336,20063,3719,1942,16,Comedy
2,vid_1002,Short Video #2,33,6,264206,37032,3228,1817,7,Food
3,vid_1003,Short Video #3,19,9,85076,27269,2371,980,1,Lifestyle
4,vid_1004,Short Video #4,47,8,90780,8041,2891,1109,23,Tech



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   video_id        300 non-null    object
 1   title           300 non-null    object
 2   duration_sec    300 non-null    int64 
 3   hashtags_count  300 non-null    int64 
 4   views           300 non-null    int64 
 5   likes           300 non-null    int64 
 6   comments        300 non-null    int64 
 7   shares          300 non-null    int64 
 8   upload_hour     300 non-null    int64 
 9   category        300 non-null    object
dtypes: int64(7), object(3)
memory usage: 23.6+ KB
None


In [30]:
df["title_length"] = df["title"].astype(str).apply(len)

In [31]:
feature_cols = ["duration_sec",
                "hashtags_count",
                "upload_hour",
                "title_length",
                "category"]

target_col = "views"

X = df[feature_cols]
y = df[target_col]


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [33]:
numeric_features = ["duration_sec", "hashtags_count", "upload_hour", "title_length"]
categorical_features = ["category"]

numeric_transformer = "passthrough"
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, categorical_features),
        ("num", numeric_transformer, numeric_features),
    ]
)

In [34]:
linreg_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

linreg_model.fit(X_train, y_train)
y_pred_lr = linreg_model.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("\n=== Linear Regression Results ===")
print("MAE:", mae_lr)
print("MSE:", mse_lr)
print("R-squared:", r2_lr)


=== Linear Regression Results ===
MAE: 126654.2829234111
MSE: 22807443768.30318
R-squared: -0.11339928507208596


In [35]:
rf_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("\n=== Random Forest Results ===")
print("MAE:", mae_rf)
print("MSE:", mse_rf)
print("R-squared:", r2_rf)


=== Random Forest Results ===
MAE: 136270.426
MSE: 25854798021.87043
R-squared: -0.26216308700233526


In [36]:
print("\n=== Model Comparison ===")
print(f"Linear Regression - MAE: {mae_lr:.2f}, R2: {r2_lr:.4f}")
print(f"Random Forest     - MAE: {mae_rf:.2f}, R2: {r2_rf:.4f}")



=== Model Comparison ===
Linear Regression - MAE: 126654.28, R2: -0.1134
Random Forest     - MAE: 136270.43, R2: -0.2622
